# Mental Health in Tech – CRISP-DM Analysis

## 1. Business Understanding

### Project Overview
Mental health issues are prevalent in the tech industry, yet many employees avoid seeking treatment due to stigma, lack of awareness, or unsupportive workplace environments. This project applies the **CRISP-DM** (Cross-Industry Standard Process for Data Mining) framework to analyze workplace mental health trends and build a classification model to predict whether a tech employee will seek mental health treatment.

### Business Problem
> **Can we predict whether a tech employee will seek mental health treatment based on workplace and demographic factors?**

Understanding the key drivers of treatment-seeking behavior can help organizations:
- Design better mental health benefit programs
- Reduce stigma through targeted awareness campaigns
- Support employees who may be at risk of untreated mental health conditions

### Success Criteria
- Achieve F1 score ≥ 0.75 on the classification task
- Identify at least 3 actionable workplace factors influencing treatment-seeking
- Deliver non-technical business recommendations

### CRISP-DM Framework
This notebook follows the standard CRISP-DM phases:
1. Business Understanding → 2. Data Understanding → 3. Data Preparation → 4. Modeling → 5. Evaluation → 6. Deployment & Recommendations

## 2. Data Understanding

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, classification_report, confusion_matrix)
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 30)
plt.rcParams['figure.dpi'] = 100
%matplotlib inline


In [ ]:
# Load the dataset
df = pd.read_csv('../data/survey.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:\n{df.columns.tolist()}")

In [ ]:
# Preview the first few rows
df.head()

In [ ]:
# Basic statistical summary of numeric columns
df.describe()

In [ ]:
# Check data types and non-null counts
df.info()

In [ ]:
# Check missing values per column
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Check value counts for the target variable
print("Target variable distribution (treatment):")
print(df['treatment'].value_counts())
print(f"\nClass balance: {df['treatment'].value_counts(normalize=True).round(3).to_dict()}")

In [ ]:
# ── Treatment Distribution Plot ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Bar chart of treatment counts
treatment_counts = df['treatment'].value_counts()
axes[0].bar(treatment_counts.index, treatment_counts.values,
            color=['#e74c3c', '#2ecc71'], edgecolor='white', alpha=0.85)
axes[0].set_title('Treatment-Seeking Counts')
axes[0].set_xlabel('Sought Treatment')
axes[0].set_ylabel('Count')
for i, v in enumerate(treatment_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart of treatment proportions
axes[1].pie(treatment_counts.values, labels=treatment_counts.index,
            autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'],
            startangle=90)
axes[1].set_title('Treatment-Seeking Proportion')

plt.suptitle('Target Variable: Treatment-Seeking Behavior', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Data Preparation

In [ ]:
# ── Step 1: Handle Missing Values ──────────────────────────────────────────
print("Missing values before cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Fill categorical missing values with 'Unknown'; keep numeric columns as-is
for col in df.select_dtypes(include='object').columns:
    df[col].fillna('Unknown', inplace=True)

print("\nMissing values after cleaning:")
print(df.isnull().sum().sum(), "total missing values")

In [ ]:
# ── Step 2: Remove Unrealistic Ages ────────────────────────────────────────
print(f"Rows before age filter: {len(df)}")

# Remove ages outside realistic range
df = df[(df['Age'] > 15) & (df['Age'] < 100)]

print(f"Rows after age filter: {len(df)}")
print(f"Age range: {df['Age'].min()} – {df['Age'].max()}")

In [ ]:
# ── Step 3: Remove Duplicates ───────────────────────────────────────────────
print(f"Rows before deduplication: {len(df)}")
df.drop_duplicates(inplace=True)
print(f"Rows after deduplication: {len(df)}")

In [ ]:
# ── Step 4: Standardize Gender Column ──────────────────────────────────────
# Normalize gender values to reduce noise
def standardize_gender(g):
    g = str(g).strip().lower()
    if g in ['male', 'm', 'man', 'cis male', 'cis man']:
        return 'Male'
    elif g in ['female', 'f', 'woman', 'cis female', 'cis woman']:
        return 'Female'
    else:
        return 'Other'

df['Gender'] = df['Gender'].apply(standardize_gender)
print("Gender distribution after standardization:")
print(df['Gender'].value_counts())

In [ ]:
# ── Step 5: Feature Engineering ─────────────────────────────────────────────
# Create age group bins
df['age_group'] = pd.cut(df['Age'], bins=[15, 25, 35, 45, 60, 100],
                         labels=['16-25', '26-35', '36-45', '46-60', '60+'])

print("Age group distribution:")
print(df['age_group'].value_counts().sort_index())

In [ ]:
# ── Step 6: Outlier Analysis (IQR Method on Age) ───────────────────────────
Q1 = df['Age'].quantile(0.25)
Q3 = df['Age'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Age'] < lower_bound) | (df['Age'] > upper_bound)]
print(f"IQR method bounds: [{lower_bound:.1f}, {upper_bound:.1f}]")
print(f"Number of age outliers detected: {len(outliers)}")

# Visualize outliers with boxplot
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(df['Age'], vert=False, patch_artist=True,
           boxprops=dict(facecolor='#3498db', alpha=0.7))
ax.set_title('Boxplot of Age – Outlier Detection (IQR Method)', fontsize=13)
ax.set_xlabel('Age')
plt.tight_layout()
plt.show()

In [ ]:
# ── Step 7: Encode Features for Modeling ────────────────────────────────────
# Select relevant features
feature_cols = ['Age', 'Gender', 'self_employed', 'family_history',
                'work_interfere', 'no_employees', 'remote_work', 'tech_company',
                'benefits', 'care_options', 'wellness_program', 'seek_help',
                'anonymity', 'mental_health_consequence', 'coworkers',
                'supervisor', 'mental_vs_physical', 'obs_consequence']

target_col = 'treatment'

df_model = df[feature_cols + [target_col]].copy()

# Label-encode all categorical features
le = LabelEncoder()
for col in df_model.select_dtypes(include='object').columns:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

print(f"Modeling dataset shape: {df_model.shape}")
df_model.head()

In [ ]:
# ── Step 8: Train / Test Split ──────────────────────────────────────────────
X = df_model.drop(columns=[target_col])
y = df_model[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size:     {X_test.shape}")
print(f"\nClass distribution in training set:\n{y_train.value_counts()}")

## 4. Modeling

### Model Selection
Since predicting mental health treatment is a **binary classification** task and the dataset may be imbalanced, we:
- Use **F1-score** as the primary evaluation metric, as it balances precision and recall
- Train three models: **Logistic Regression** (baseline), **Random Forest** (ensemble), and **Decision Tree**
- Apply **cross-validation** to get robust performance estimates
- Use **GridSearchCV** for hyperparameter tuning of both Random Forest and Decision Tree


In [ ]:
# ── Baseline: Logistic Regression (Pipeline with StandardScaler) ────────────
lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)

print('=== Logistic Regression (Pipeline: StandardScaler + LR) ===')
print(f'Accuracy:  {lr_accuracy:.4f}')
print(f'Precision: {lr_precision:.4f}')
print(f'Recall:    {lr_recall:.4f}')
print(f'F1 Score:  {lr_f1:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr, target_names=['No Treatment', 'Treatment']))

In [ ]:
# ── Random Forest with GridSearchCV ─────────────────────────────────────────
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)

print("=== Random Forest (GridSearchCV best params) ===")
print(f"Best params: {rf_grid.best_params_}")
print(f"\nAccuracy:  {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall:    {rf_recall:.4f}")
print(f"F1 Score:  {rf_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['No Treatment', 'Treatment']))

In [ ]:
# ── Decision Tree Classifier ─────────────────────────────────────────────────
dt_param_grid = {
    'max_depth': [5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)
dt_grid.fit(X_train, y_train)

best_dt = dt_grid.best_estimator_
y_pred_dt = best_dt.predict(X_test)

dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt)
dt_precision = precision_score(y_test, y_pred_dt)
dt_recall = recall_score(y_test, y_pred_dt)

print("=== Decision Tree (GridSearchCV best params) ===")
print(f"Best params: {dt_grid.best_params_}")
print(f"\nAccuracy:  {dt_accuracy:.4f}")
print(f"Precision: {dt_precision:.4f}")
print(f"Recall:    {dt_recall:.4f}")
print(f"F1 Score:  {dt_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt, target_names=['No Treatment', 'Treatment']))


In [ ]:
# ── Cross-Validation for All Models ─────────────────────────────────────────
lr_cv_scores = cross_val_score(lr_model, X, y, cv=5, scoring='f1')
rf_cv_scores = cross_val_score(best_rf, X, y, cv=5, scoring='f1')
dt_cv_scores = cross_val_score(best_dt, X, y, cv=5, scoring='f1')

print("Cross-Validation F1 Scores (5-fold):")
print(f"Logistic Regression: {lr_cv_scores.mean():.4f} ± {lr_cv_scores.std():.4f}")
print(f"Random Forest:       {rf_cv_scores.mean():.4f} ± {rf_cv_scores.std():.4f}")
print(f"Decision Tree:       {dt_cv_scores.mean():.4f} ± {dt_cv_scores.std():.4f}")


In [ ]:
# ── Feature Importance Plot ──────────────────────────────────────────────────
importances = best_rf.feature_importances_
feature_names = X.columns

feat_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feat_df, palette='viridis', ax=ax)
ax.set_title('Top 10 Feature Importances – Random Forest', fontsize=14)
ax.set_xlabel('Importance Score')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

## 5. Evaluation

### Why F1-Score?
We use **F1-score** because the dataset may be imbalanced and we want to balance **precision** (avoid false alarms) and **recall** (catch those who need help). In healthcare contexts, missing someone who needs treatment (false negative) is costly, making F1 the most meaningful single metric.

### Results Summary

In [ ]:
# ── Model Comparison Summary ─────────────────────────────────────────────────
results = {
    'Model': ['Logistic Regression', 'Random Forest', 'Decision Tree'],
    'Accuracy': [lr_accuracy, rf_accuracy, dt_accuracy],
    'Precision': [lr_precision, rf_precision, dt_precision],
    'Recall': [lr_recall, rf_recall, dt_recall],
    'F1 Score': [lr_f1, rf_f1, dt_f1]
}
results_df = pd.DataFrame(results)
results_df = results_df.set_index('Model')
print(results_df.round(4))

best_model_name = results_df['F1 Score'].idxmax()
best_model_f1 = results_df['F1 Score'].max()
print(f"\n✅ {best_model_name} performed best with F1 = {best_model_f1:.4f}")


In [ ]:
# ── Confusion Matrices ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, y_pred, title in zip(
    axes,
    [y_pred_lr, y_pred_rf, y_pred_dt],
    ['Logistic Regression', 'Random Forest', 'Decision Tree']
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Treatment', 'Treatment'],
                yticklabels=['No Treatment', 'Treatment'])
    ax.set_title(f'Confusion Matrix – {title}', fontsize=12)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Model Evaluation – Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── Key Interpretations ─────────────────────────────────────────────────────
print("Key Model Interpretations:")
print("-" * 50)
fh_counts = df.groupby(['family_history', 'treatment']).size().unstack(fill_value=0)
fh_pct = fh_counts.div(fh_counts.sum(axis=1), axis=0) * 100
print("\nTreatment rate by family history:")
print(fh_pct.round(1))

print("\nTreatment rate by benefits:")
ben_counts = df.groupby(['benefits', 'treatment']).size().unstack(fill_value=0)
ben_pct = ben_counts.div(ben_counts.sum(axis=1), axis=0) * 100
print(ben_pct.round(1))

### Exploratory Data Analysis – Visualizations

In [ ]:
# ── Age Distribution ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram
axes[0].hist(df['Age'], bins=30, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_title('Age Distribution of Survey Respondents')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

# Boxplot: Age vs Treatment
sns.boxplot(x='treatment', y='Age', data=df, palette='Set2', ax=axes[1])
axes[1].set_title('Boxplot: Age vs Treatment-Seeking')
axes[1].set_xlabel('Sought Treatment')
axes[1].set_ylabel('Age')

plt.suptitle('Age Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Categorical Plots ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Treatment vs Gender
sns.countplot(x='Gender', hue='treatment', data=df, palette='Set1', ax=axes[0])
axes[0].set_title('Treatment by Gender')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Count')
axes[0].legend(title='Treatment')

# Treatment vs Company Size
sns.countplot(x='no_employees', hue='treatment', data=df, palette='Set2', ax=axes[1])
axes[1].set_title('Treatment by Company Size')
axes[1].set_xlabel('Company Size')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Treatment')

# Treatment vs Benefits
sns.countplot(x='benefits', hue='treatment', data=df, palette='Set3', ax=axes[2])
axes[2].set_title('Treatment by Mental Health Benefits')
axes[2].set_xlabel('Benefits Available')
axes[2].set_ylabel('Count')
axes[2].legend(title='Treatment')

plt.suptitle('Treatment-Seeking Behavior by Workplace Factors', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation Heatmap ──────────────────────────────────────────────────────
numeric_cols = df_model.select_dtypes(include='number').columns
corr_matrix = df_model[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Age Group vs Treatment ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(x='age_group', hue='treatment', data=df, palette='husl', ax=ax)
ax.set_title('Treatment-Seeking by Age Group')
ax.set_xlabel('Age Group')
ax.set_ylabel('Count')
ax.legend(title='Treatment')
plt.tight_layout()
plt.show()

## 6. Deployment & Recommendations

### Model Deployment Strategy
The best-performing model (Random Forest) can be deployed as an internal HR analytics tool to:
- Identify employee segments with lower treatment-seeking likelihood
- Trigger targeted wellness outreach programs
- Monitor changes in workplace mental health metrics over time

### 🏥 Business Recommendations

Based on the analysis and model results, we recommend the following non-technical actions:

#### 1. **Improve Mental Health Benefit Awareness**
Employees who are aware of and have access to mental health benefits are significantly more likely to seek treatment. Organizations should:
- Communicate mental health benefits clearly during onboarding and annually
- Simplify the process for accessing mental health services

#### 2. **Ensure Confidentiality and Anonymity**
Anonymity in reporting is a strong predictor of treatment-seeking. Companies should:
- Guarantee that mental health disclosures remain confidential
- Create anonymous reporting channels and wellness surveys

#### 3. **Address Family History Stigma**
Employees with a family history of mental illness are significantly more likely to seek treatment, suggesting awareness already exists in that group. For others:
- Normalize mental health conversations through company culture initiatives
- Educate employees about the hereditary nature of mental health conditions

#### 4. **Train Supervisors on Mental Health Communication**
Supervisor support strongly influences whether employees feel comfortable disclosing mental health challenges. Recommended actions:
- Provide mental health first aid training for all managers
- Create a culture where supervisors check in on employee wellbeing regularly

#### 5. **Target Remote Workers Differently**
Remote employees may have different access to mental health resources. Organizations should:
- Offer virtual mental health support options
- Conduct regular check-ins for remote team members

### Summary of Key Findings
| Finding | Recommendation |
|---------|----------------|
| Family history is the strongest predictor | Increase awareness for all employees, not just those with family history |
| Benefits availability increases treatment rates | Improve benefit communication and accessibility |
| Workplace anonymity matters | Protect confidentiality of mental health disclosures |
| Supervisor support is critical | Train managers on mental health communication |
| Work interference is highly predictive | Offer flexible work arrangements and EAP programs |

In [ ]:
# ── Final Summary Print ──────────────────────────────────────────────────────
model_scores = {
    'Logistic Regression': (lr_accuracy, lr_f1, lr_precision, lr_recall),
    'Random Forest': (rf_accuracy, rf_f1, rf_precision, rf_recall),
    'Decision Tree': (dt_accuracy, dt_f1, dt_precision, dt_recall),
}
best_name = max(model_scores, key=lambda k: model_scores[k][1])
best_acc, best_f1, best_pre, best_rec = model_scores[best_name]

print("=" * 60)
print("MENTAL HEALTH IN TECH – CRISP-DM ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nDataset: {len(df)} employees surveyed")
print(f"Target: Treatment-seeking behavior (Yes/No)")
print(f"\nModel Performance:")
for name, (acc, f1, pre, rec) in model_scores.items():
    print(f"  {name:<25} Accuracy={acc:.4f}  F1={f1:.4f}")
print(f"\nBest Model: {best_name}")
print(f"  Accuracy:  {best_acc:.4f}")
print(f"  F1 Score:  {best_f1:.4f}")
print(f"  Precision: {best_pre:.4f}")
print(f"  Recall:    {best_rec:.4f}")
print(f"\nTop Predictor: family_history, work_interfere, benefits")
print(f"\nKey Finding: Employees with family history of mental illness")
print(f"are significantly more likely to seek treatment.")
print(f"\nRecommendation: Invest in benefit awareness, supervisor")
print(f"training, and anonymous mental health support channels.")
print("=" * 60)
